In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


In [2]:
from collections import defaultdict
from copy import deepcopy
from uuid import uuid4

from src.utils import disk
from src.utils import debug

In [4]:
from src.processing.ids import id_mapper, IDMapper

In [3]:
pr = 'data/samples/prospects.json'
dr = 'data/samples_nbaapi/drafts.json'
dh = 'data/samples_nbaapi/drafthistory.json'

pr = disk.read_json(pr)
dr = disk.read_json(dr)
dh = disk.read_json(dh)

In [6]:
def construct_college_team_dimension(pr: dict, id_mapper: IDMapper) -> tuple[list, list, list, dict]:
    # TODO: Ignored fields draft/venue, draft/start_date, draft/end_date, draft/league

    pr = deepcopy(pr)
    
    list_of_prospects: list[dict] = pr['prospects']

    draft_year = pr['draft']['year']

    college_teams = []
    prospects = []

    # --------------------------
    # Helper function to construct entries for conference, division, and team
    # --------------------------
    def to_entry(namespace: str, prefix: str, obj: dict, mappings: str, provider: str, provider_id: str) -> dict:
        entry = {}
        alias = obj.get('alias')
        kwargs = {k: obj.get(k) for k in mappings}

        if alias:
            obj_id = id_mapper(namespace, alias, provider, provider_id, str(uuid4()))
            entry = {
                f'{prefix}_id': obj_id,
                f'{prefix}_alias': alias,
                **{f'{prefix}_{k}': v for k, v in kwargs.items()}
            }

        return entry

    for prospect in list_of_prospects:
        # --------------------------
        # Construct entries for conference, division, and team
        # --------------------------
        conference: dict = prospect.pop('conference', {})
        division: dict = prospect.pop('division', {})
        team: dict = prospect.pop('team', {})

        conference_entry = to_entry(
            namespace='conference',
            prefix='conference',
            obj=conference,
            mappings=('name', ),
            provider='sportsradar',
            provider_id=conference.get('id'),
        )
        
        division_entry = to_entry(
            namespace='division',
            prefix='division',
            obj=division,
            mappings=('name', ),
            provider='sportsradar',
            provider_id=division.get('id'),
        )

        team_entry = to_entry(
            namespace='team',
            prefix='team',
            obj=team,
            mappings=('name', 'market'),
            provider='sportsradar',
            provider_id=team.get('id'),
        )

        college_teams.append({
            **team_entry, 
            **division_entry, 
            **conference_entry
        })

        # --------------------------
        # Construct prospect entry
        # --------------------------
        prospect.pop('source_id', None)

        college_team_entry = college_teams[-1]

        birthplace = prospect.pop('birth_place', '').split(',')
        birthplace = zip(('city', 'state', 'country'), birthplace)
        birthplace = {f'birth_{k}':v.strip() or None for k, v in birthplace}
        
        p_id = id_mapper('prospect', prospect['name'], 'sportsradar', prospect.pop('id'), str(uuid4()))
        
        prospects.append({
            'prospect_id': p_id,
            'draft_year': draft_year,
            **prospect,
            **birthplace,
            'team_id': college_team_entry.get('team_id'),
        })

    return college_teams, prospects



In [7]:
college_teams, prospects = construct_college_team_dimension(pr, id_mapper)

In [8]:
import pandas as pd

df_prospects = pd.DataFrame(prospects)
df_college_teams = pd.DataFrame(college_teams)

df_prospects.head()

,prospect_id,draft_year,first_name,last_name,name,position,height,weight,experience,team_name,birth_city,birth_state,birth_country,team_id,high_school,top_prospect,league_id
0,15ba374a-5eb1-4fe3-92c8-1598ad2ec57f,2025,Brice,Williams,Brice Williams,G,79,214.0,SR,Nebraska,Huntersville,NC,USA,4b9d22e0-3e9e-494f-9c25-43ac0013ba04,NaN,NaN,NaN
1,0657dbf1-d4c5-476b-94ed-c012029fb3ef,2025,Jalon,Moore,Jalon Moore,F,79,215.0,SR,Oklahoma,Birmingham,AL,USA,2ceba0d0-c203-44ff-aba6-964e957b09e3,NaN,NaN,NaN
2,0fcfee3f-5be4-44d8-951b-23d59a0b77d1,2025,Tamar,Bates,Tamar Bates,G,77,195.0,SR,Missouri,Kansas City,KS,USA,978f4a04-bb82-4e39-873d-d7d3ab600b59,NaN,NaN,NaN
3,201386d6-0e8e-44b1-af6d-cafcdae69fae,2025,Grant,Nelson,Grant Nelson,F,83,230.0,GR,Alabama,Devils Lake,ND,USA,33a42bc1-10cc-4f45-ac2e-465ec723c63a,NaN,NaN,NaN
4,31922f01-6927-428e-87ef-899d706262fc,2025,Jaxson,Robinson,Jaxson Robinson,G,78,192.0,GR,Kentucky,Ada,OK,USA,83fde951-20d6-428c-aa4a-aa25b8be200e,NaN,NaN,NaN


In [9]:
df_college_teams.head()

,team_id,team_alias,team_name,team_market,division_id,division_alias,division_name,conference_id,conference_alias,conference_name
0,4b9d22e0-3e9e-494f-9c25-43ac0013ba04,NEB,Cornhuskers,Nebraska,acfc135d-ffe3-4ee7-bffd-564a591630dc,D1,NCAA Division I,ae0c81f7-08ea-4e94-bc18-19ceab0599a5,BIG10,Big Ten
1,2ceba0d0-c203-44ff-aba6-964e957b09e3,OKLA,Sooners,Oklahoma,acfc135d-ffe3-4ee7-bffd-564a591630dc,D1,NCAA Division I,94c9be59-9d74-4c01-86b8-ed96de94dabc,SEC,Southeastern
2,978f4a04-bb82-4e39-873d-d7d3ab600b59,MIZZ,Tigers,Missouri,acfc135d-ffe3-4ee7-bffd-564a591630dc,D1,NCAA Division I,94c9be59-9d74-4c01-86b8-ed96de94dabc,SEC,Southeastern
3,33a42bc1-10cc-4f45-ac2e-465ec723c63a,ALA,Crimson Tide,Alabama,acfc135d-ffe3-4ee7-bffd-564a591630dc,D1,NCAA Division I,94c9be59-9d74-4c01-86b8-ed96de94dabc,SEC,Southeastern
4,83fde951-20d6-428c-aa4a-aa25b8be200e,UK,Wildcats,Kentucky,acfc135d-ffe3-4ee7-bffd-564a591630dc,D1,NCAA Division I,94c9be59-9d74-4c01-86b8-ed96de94dabc,SEC,Southeastern


## A few problems
- the prospect table above
    - does not currently have ids mapping to an actual player
    - does not currently cover all players
    - only has data dating back to 2019
- the college team table above
    - does not account for other kinds of affiliate organizations, such as foreign teams

In [10]:
draft = disk.read_json('data/samples_nbaapi/drafthistory.json')
draft = draft['resultSets'][0]
draft = pd.DataFrame(draft['rowSet'], columns=draft['headers'])
draft.columns = draft.columns.str.lower()

In [34]:
# In total we have 8127 prospects
# NBA API provides 8060
# Sportsradar provides 121
# They have 54 prospects in common (intersection)
# Whilst differing in 8073 prospects (total - intersection) 

set_a = set(draft.player_name.str.lower().values)
set_b = set(df_prospects.name.str.lower().values)

diff = set_a.difference(set_b)
same = set_a.intersection(set_b)
total = set_a | set_b

len(total-same), len(same), len(set_a), len(set_b), len(total)

(8073, 54, 8060, 121, 8127)

In [23]:
from src.clients import connect_mysql

In [24]:
conn = connect_mysql()

In [27]:
dim_players = pd.read_sql('SELECT * FROM dim_player', conn)
fct_players = pd.read_sql('SELECT game_id, player_id FROM fact_player_game_stats', conn)

/tmp/ipykernel_485789/1111650292.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_players = pd.read_sql('SELECT * FROM dim_player', conn)
/tmp/ipykernel_485789/1111650292.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fct_players = pd.read_sql('SELECT game_id, player_id FROM fact_player_game_stats', conn)


In [62]:
dim_players['full_name'] = (dim_players.first_name.str.lower() + ' ' + dim_players.family_name.str.lower()).str.strip()

dim_players_subset = dim_players[dim_players.full_name != '']

In [29]:
dim_players.player_id.nunique(), fct_players.player_id.nunique() # Consistent number of players in dim and fact table 

(4691, 4691)

In [91]:
def set_comparison(x, y):
    set_x = set(x)
    set_y = set(y)

    total = set_x | set_y
    same = set_x.intersection(set_y)
    diff = total - same

    return dict(
        x=len(set_x),
        y=len(set_y),
        x_diff_y=len(set_x - set_y),
        y_diff_x=len(set_y - set_x),
        total=len(total),
        same=len(same),
        diff=len(diff)
    )


In [63]:
# In total we have 8127 prospects
# NBA API provides 8060
# Sportsradar provides 121
# They have 54 prospects in common (intersection)
# Whilst differing in 8073 prospects (total - intersection) 

# Approximately 1k players do not exist in reality
# Multiple players have the same name, but different player_ids (e.g. Isaiah Thomas)


a = draft.player_name.str.lower().values
b = dim_players_subset.full_name.values

debug.prettyprint(set_comparison(a, b))

{
    "x": 8060,
    "y": 3518,
    "total": 9839,
    "same": 1739,
    "diff": 8100
}


In [64]:

a = draft.person_id.astype(str).values
b = dim_players_subset.player_id.astype(str).values

debug.prettyprint(set_comparison(a, b))

{
    "x": 8235,
    "y": 3685,
    "total": 10192,
    "same": 1728,
    "diff": 8464
}


In [67]:
3518, 3685-167

(3518, 3518)

In [77]:
# dim_players_subset.groupby('full_name')['player_id'].apply(list).to_dict()

In [50]:
x = dim_players[dim_players.full_name == '']

In [58]:
# tmp = fct_players[fct_players.player_id.isin(x.player_id)].groupby('player_id')['game_id'].apply(list).to_dict()

# disk.write_json('players_with_no_name.json', tmp)

In [72]:
player_index = disk.read_json('data/_v2/player_index.json')

In [85]:
player_movement = disk.read_json('data/_v2/player_movement.json')
y = player_movement['NBA_Player_Movement']
y = pd.DataFrame(y['rows'])
y.Transaction_Type.value_counts()

Transaction_Type
Signing              4411
Waive                3125
Trade                1724
ContractConverted     127
AwardOnWaivers         69
Name: count, dtype: int64

In [90]:
y

,Transaction_Type,TRANSACTION_DATE,TRANSACTION_DESCRIPTION,TEAM_ID,TEAM_SLUG,PLAYER_ID,PLAYER_SLUG,Additional_Sort,GroupSort
0,Signing,2026-04-10T00:00:00,Memphis Grizzlies signed guard Lucas Williamso...,1.610613e+09,grizzlies,1631351.0,lucas-williamson,0.0,Signing 1148495
1,Signing,2026-04-09T00:00:00,Chicago Bulls signed forward Mouhamadou Gueye ...,1.610613e+09,bulls,1631338.0,mouhamadou-gueye,0.0,Signing 1148489
2,Signing,2026-04-07T00:00:00,Memphis Grizzlies signed guard Adama Bal to a ...,1.610613e+09,grizzlies,1642380.0,adama-bal,0.0,Signing 1148392
3,Signing,2026-04-07T00:00:00,Houston Rockets re-signed guard JD Davison to ...,1.610613e+09,rockets,1631120.0,jd-davison,0.0,Signing 1148393
4,Signing,2026-04-07T00:00:00,Detroit Pistons re-signed forward Tolu Smith t...,1.610613e+09,pistons,1642449.0,tolu-smith,0.0,Signing 1148402
...,...,...,...,...,...,...,...,...,...
9451,Signing,2015-07-02T00:00:00,Brooklyn Nets signed guard Ryan Boatright to a...,1.610613e+09,nets,1626207.0,ryan-boatright,0.0,Signing 944820
9452,Signing,2015-07-02T00:00:00,Charlotte Hornets re-signed forward Frank Kami...,1.610613e+09,hornets,1626163.0,frank-kaminsky,0.0,Signing 944876
9453,Signing,2015-07-02T00:00:00,Minnesota Timberwolves re-signed guard Tyus Jo...,1.610613e+09,timberwolves,1626145.0,tyus-jones,0.0,Signing 944877
9454,Signing,2015-07-02T00:00:00,Toronto Raptors re-signed guard Delon Wright t...,1.610613e+09,raptors,1626153.0,delon-wright,0.0,Signing 944878


In [74]:
x = player_index['resultSets'][-1]
x = pd.DataFrame(x['rowSet'], columns=x['headers'])
x

,PERSON_ID,PLAYER_LAST_NAME,PLAYER_FIRST_NAME,PLAYER_SLUG,TEAM_ID,TEAM_SLUG,IS_DEFUNCT,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,...,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER,ROSTER_STATUS,PTS,REB,AST,STATS_TIMEFRAME,FROM_YEAR,TO_YEAR
0,76001,Abdelnaby,Alaa,alaa-abdelnaby,1610612757,blazers,0,Portland,Trail Blazers,POR,...,1990.0,1.0,25.0,NaN,5.7,3.3,0.3,Career,1990,1994
1,76002,Abdul-Aziz,Zaid,zaid-abdul-aziz,1610612745,rockets,0,Houston,Rockets,HOU,...,1968.0,1.0,5.0,NaN,9.0,8.0,1.2,Career,1968,1977
2,76003,Abdul-Jabbar,Kareem,kareem-abdul-jabbar,1610612747,lakers,0,Los Angeles,Lakers,LAL,...,1969.0,1.0,1.0,NaN,24.6,11.2,3.6,Career,1969,1988
3,51,Abdul-Rauf,Mahmoud,mahmoud-abdul-rauf,1610612743,nuggets,0,Denver,Nuggets,DEN,...,1990.0,1.0,3.0,NaN,14.6,1.9,3.5,Career,1990,2000
4,1505,Abdul-Wahad,Tariq,tariq-abdul-wahad,1610612758,kings,0,Sacramento,Kings,SAC,...,1997.0,1.0,11.0,NaN,7.8,3.3,1.1,Career,1997,2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5121,78648,Zopf,Bill,bill-zopf,1610612749,bucks,0,Milwaukee,Bucks,MIL,...,1970.0,2.0,33.0,NaN,2.2,0.9,1.4,Career,1970,1970
5122,1627826,Zubac,Ivica,ivica-zubac,1610612754,pacers,0,Indiana,Pacers,IND,...,2016.0,2.0,32.0,1.0,14.1,10.6,2.2,Season,2016,2025
5123,78650,Zunic,Matt,matt-zunic,1610610036,None,1,Washington,Capitols,WAS,...,1947.0,NaN,NaN,NaN,4.9,NaN,0.9,Career,1948,1948
5124,1641783,da Silva,Tristan,tristan-da-silva,1610612753,magic,0,Orlando,Magic,ORL,...,2024.0,1.0,18.0,1.0,10.0,3.7,1.5,Season,2024,2025


In [81]:
x[x.PERSON_ID==1504]

,PERSON_ID,PLAYER_LAST_NAME,PLAYER_FIRST_NAME,PLAYER_SLUG,TEAM_ID,TEAM_SLUG,IS_DEFUNCT,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,...,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER,ROSTER_STATUS,PTS,REB,AST,STATS_TIMEFRAME,FROM_YEAR,TO_YEAR
1445,1504,Fortson,Danny,danny-fortson,1610612743,nuggets,0,Denver,Nuggets,DEN,...,1997.0,1.0,10.0,NaN,8.2,7.2,0.7,Career,1997,2006


In [79]:
x.FROM_YEAR.agg(['min', 'max'])

min    1946
max    2025
Name: FROM_YEAR, dtype: object

In [92]:
a = dim_players_subset.player_id.astype(str).values
b = x.PERSON_ID.astype(str).values
debug.prettyprint(set_comparison(a, b))

{
    "x": 3685,
    "y": 5126,
    "x_diff_y": 1161,
    "y_diff_x": 2602,
    "total": 6287,
    "same": 2524,
    "diff": 3763
}


In [95]:
diff = set(a).difference(set(b))
mask = dim_players_subset.player_id.isin(diff)
dim_players_subset[mask]

,player_id,first_name,family_name,name_initial,player_slug,full_name
48,101151,Erazem,Lorbek,E. Lorbek,erazem-lorbek,erazem lorbek
88,101217,D'or,Fischer,D. Fischer,dor-fischer,d'or fischer
92,101224,Deji,Akindele,D. Akindele,deji-akindele,deji akindele
103,101239,Otis,George,O. George,otis-george,otis george
106,101243,Chris,Alexander,C. Alexander,chris-alexander,chris alexander
...,...,...,...,...,...,...
4556,7020,Dillon,Stith,D. Stith,dillon-stith,dillon stith
4557,7021,Tohi,Smith-Milner,T. Smith-Milner,tohi-smith-milner,tohi smith-milner
4600,8013,Ricky,Rubio,R. Rubio,ricky-rubio,ricky rubio
4624,9007,Boban,Marjanovic,B. Marjanovic,boban-marjanovic,boban marjanovic


In [107]:
nba_teams = disk.read_json('data/_v2_dimensions/nba_teams.json')
nba_teams = pd.DataFrame(nba_teams)

In [158]:
nba_teams

,team_id,team_name,team_alias,team_market,division_id,division_name,division_alias,conference_id,conference_name,conference_alias,founded_in,venue_name,venue_capacity,venue_address,venue_city,venue_state,venue_zip,venue_country
0,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,Washington,59b168e6-8c4c-4c42-8fc1-784c057212c0,Southeast,SOUTHEAST,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1961,Capital One Arena,20356,601 F Street NW,Washington,DC,20004,USA
1,e760d6ee-350f-45cb-891d-bdeed1aa9f98,Hornets,CHA,Charlotte,59b168e6-8c4c-4c42-8fc1-784c057212c0,Southeast,SOUTHEAST,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1988,Spectrum Center,19077,330 E. Trade Street,Charlotte,NC,28202,USA
2,a4b5ae6b-041c-4af8-b442-05a2bb9e1256,Hawks,ATL,Atlanta,59b168e6-8c4c-4c42-8fc1-784c057212c0,Southeast,SOUTHEAST,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1946,State Farm Arena,18118,One Philips Drive,Atlanta,GA,30303,USA
3,ffc2f02b-e90f-4808-9b3a-02d5fb3814b4,Heat,MIA,Miami,59b168e6-8c4c-4c42-8fc1-784c057212c0,Southeast,SOUTHEAST,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1988,Kaseya Center,19600,601 Biscayne Boulevard,Miami,FL,33132,USA
4,6f2d061b-e1f6-431f-898a-088b65c5ca63,Magic,ORL,Orlando,59b168e6-8c4c-4c42-8fc1-784c057212c0,Southeast,SOUTHEAST,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1989,Kia Center,18846,400 W. Church Street,Orlando,FL,32801,USA
5,7115bb5c-70c6-4b16-a9fb-e87d9add68fd,Knicks,NYK,New York,ea2534c6-96bb-4bee-9dae-a284302212cc,Atlantic,ATLANTIC,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1946,Madison Square Garden,19812,4 Pennsylvania Plaza,New York,NY,10001,USA
6,753a9d2b-7d27-4e56-88d7-87e426c73204,76ers,PHI,Philadelphia,ea2534c6-96bb-4bee-9dae-a284302212cc,Atlantic,ATLANTIC,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1946,Xfinity Mobile Arena,20478,3601 S. Broad Street,Philadelphia,PA,19148,USA
7,39a12a6b-baf2-4969-8a8a-81746f2b72c5,Nets,BKN,Brooklyn,ea2534c6-96bb-4bee-9dae-a284302212cc,Atlantic,ATLANTIC,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1967,Barclays Center,17732,620 Atlantic Avenue,Brooklyn,NY,11217,USA
8,94b97ffb-d045-456e-bca2-3aca2fbb1092,Celtics,BOS,Boston,ea2534c6-96bb-4bee-9dae-a284302212cc,Atlantic,ATLANTIC,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1946,TD Garden,18624,100 Legends Way,Boston,MA,02114,USA
9,1dd3b092-1e11-4de1-95de-1db1639c3608,Raptors,TOR,Toronto,ea2534c6-96bb-4bee-9dae-a284302212cc,Atlantic,ATLANTIC,2a2cb4c5-103b-4f23-b620-576b2e7200fa,EASTERN CONFERENCE,EASTERN,1995,Scotiabank Arena,19800,40 Bay Street,Toronto,ON,M5J 2X2,CAN


In [98]:
game_ = pd.read_sql('SELECT * FROM dim_game', conn)

/tmp/ipykernel_485789/3770741270.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  game_ = pd.read_sql('SELECT * FROM dim_game', conn)


In [159]:
# game_

,game_id,game_date,home_team_tricode,away_team_tricode,home_team_id,away_team_id,season_label,season_type
0,0010300001,2003-10-05,UTA,DAL,1610612762,1610612742,2003-04,Pre_Season
1,0010300002,2003-10-06,MEM,MIL,1610612763,1610612749,2003-04,Pre_Season
2,0010300003,2003-10-07,DET,CLE,1610612765,1610612739,2003-04,Pre_Season
3,0010300004,2003-10-07,DAL,ORL,1610612742,1610612753,2003-04,Pre_Season
4,0010300005,2003-10-07,POR,HOU,1610612757,1610612745,2003-04,Pre_Season
...,...,...,...,...,...,...,...,...
35341,0052400111,2025-04-16,CHI,MIA,1610612741,1610612748,2024-25,PlayIn
35342,0052400121,2025-04-15,GSW,MEM,1610612744,1610612763,2024-25,PlayIn
35343,0052400131,2025-04-16,SAC,DAL,1610612758,1610612742,2024-25,PlayIn
35344,0052400201,2025-04-18,ATL,MIA,1610612737,1610612748,2024-25,PlayIn


In [133]:
game_.home_team_id = game_.home_team_id.apply(lambda x: int(float(x)))
game_.away_team_id = game_.away_team_id.apply(lambda x: int(float(x)))

In [152]:
team_tricodes_nba = set(nba_teams.team_alias.unique())
home_team_mask = game_.home_team_tricode.isin(team_tricodes_nba)
away_team_mask = game_.away_team_tricode.isin(team_tricodes_nba)

home_team_only = game_[home_team_mask][['home_team_id', 'home_team_tricode']].drop_duplicates().set_index('home_team_id')


home_team = game_[['home_team_id', 'home_team_tricode']].drop_duplicates()
related_tricodes = home_team.groupby('home_team_id')['home_team_tricode'].apply(list)

In [164]:
mask = ~related_tricodes.index.isin(home_team_only.index)
related_tricodes[mask]

home_team_id
45                      [CHN]
94                 [MLN, EAM]
12303                   [MTA]
12304              [BAR, FCB]
12305                   [LYV]
12306                   [KHI]
12307                   [ROM]
12308                   [MOS]
12309                   [EPT]
12311                   [MAL]
12315              [RMA, RMD]
12317                   [LRO]
12318                   [MMT]
12321                   [FBU]
12323                   [ALB]
12324                   [UBB]
12325                   [FLA]
1610616833    [EST, GNS, DRT]
1610616834    [WST, STP, LBN]
Name: home_team_tricode, dtype: object

In [ ]:
pd.DataFrame.

In [165]:
home_team_only['related_tricodes'] = related_tricodes.loc[home_team_only.index]
home_team_only.sort_index()

,home_team_tricode,related_tricodes
home_team_id,,
1610612737,ATL,[ATL]
1610612738,BOS,[BOS]
1610612739,CLE,[CLE]
1610612740,NOP,"[NOH, NOK, NOP]"
1610612741,CHI,[CHI]
1610612742,DAL,[DAL]
1610612743,DEN,[DEN]
1610612744,GSW,[GSW]
1610612745,HOU,[HOU]


In [160]:
len(home_team_only)

30

In [143]:
1610612766 - 1610612737

29

In [119]:
team_tricodes_nba = set(nba_teams.team_alias.unique())
team_tricodes_other = set(game_.home_team_tricode.unique()) | set(game_.away_team_tricode.unique())
team_tricodes_other = team_tricodes_other - team_tricodes_nba
len(team_tricodes_other), len(team_tricodes_nba)

(58, 30)

In [121]:
mask = game_.home_team_tricode.isin(team_tricodes_other) | game_.away_team_tricode.isin(team_tricodes_other)
game_[mask]

,game_id,game_date,home_team_tricode,away_team_tricode,home_team_id,away_team_id,season_label,season_type
6,0010300008,2003-10-07,PHX,NJN,1610612756,1610612751,2003-04,Pre_Season
12,0010300016,2003-10-08,SEA,HOU,1610612760,1610612745,2003-04,Pre_Season
13,0010300017,2003-10-08,NOH,ORL,1610612740,1610612753,2003-04,Pre_Season
19,0010400068,2004-10-22,BOS,NJN,1610612738.0,1610612751.0,2004-05,Pre_Season
26,0010400075,2004-10-23,SAC,NOH,1610612758.0,1610612740.0,2004-05,Pre_Season
...,...,...,...,...,...,...,...,...
34088,0041000152,2011-04-20,LAL,NOH,1610612747,1610612740,2010-11,Playoffs
34089,0041000153,2011-04-22,NOH,LAL,1610612740,1610612747,2010-11,Playoffs
34090,0041000154,2011-04-24,NOH,LAL,1610612740,1610612747,2010-11,Playoffs
34091,0041000155,2011-04-26,LAL,NOH,1610612747,1610612740,2010-11,Playoffs


In [97]:
mask = fct_players.player_id.isin(diff)
fct_players[mask]

,game_id,player_id
37,0011300011,203568
39,0011300011,202586
40,0011300011,202518
54,0011300011,203577
755,0011000025,202402
...,...,...
828183,0011900014,1629595
828204,0011900014,1629609
828205,0011900014,1629722
828206,0011900014,1629777
